In [3]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import accuracy_score, brier_score_loss, classification_report, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline

In [5]:

# =========================
# 1️⃣ Load Dataset (تحميل البيانات)
# =========================
DATA_PATH = "dataset/Final_Augmented_dataset_Diseases_and_Symptoms.csv"
try:
    df = pd.read_csv(DATA_PATH)
    print("✅ Dataset Loaded. Original Shape:", df.shape)
except FileNotFoundError:
    print("❌ Error: File not found. Please check the path.")
    exit()


✅ Dataset Loaded. Original Shape: (246945, 378)


In [6]:
# =========================
# 2️⃣ Cleaning (التنظيف)
# =========================
df = df.drop_duplicates()
print("After Removing Duplicates:", df.shape)

After Removing Duplicates: (189647, 378)


In [7]:
# =========================
# 3️⃣ Remove rare classes (إزالة الفئات النادرة)
# =========================
class_counts = df["diseases"].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df["diseases"].isin(valid_classes)]
print("After Removing Rare Classes:", df.shape)

After Removing Rare Classes: (189601, 378)


In [8]:
# =========================
# 4️⃣ Split Features / Target
# =========================
X = df.drop("diseases", axis=1).astype("int8")
y = df["diseases"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [9]:
# =========================
# 4️⃣ Split Features / Target (فصل الميزات والهدف)
# =========================
X = df.drop("diseases", axis=1).astype("int8")
y = df["diseases"]
# تشفير المخرجات (الأمراض)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [10]:
# =========================
# 5️⃣ Train / Test Split (تقسيم البيانات)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)


In [11]:

# =========================
# 6️⃣ Define Pipeline & Model (تعريف البايبلاين والنموذج)
# =========================
print("🛠️ Building Pipeline...")

# تعريف النموذج الأساسي
rf = RandomForestClassifier(
    n_estimators=100,         # زيادة العدد قليلاً لتحسين الدقة
    max_depth=20,             
    min_samples_leaf=2,
    n_jobs=-1,                # استخدام كل المعالجات لتسريع العمل
    random_state=42,
    class_weight="balanced"
)
# تعريف المعايرة (Calibration)
calibrated_rf = CalibratedClassifierCV(
    estimator=rf,
    method="isotonic",
    cv=2
)
model_pipeline = Pipeline([
    ('classifier', calibrated_rf)
])

🛠️ Building Pipeline...


In [12]:
# =========================
# 7️⃣ Train the Pipeline (تدريب البايبلاين)
# =========================
print("🌲 Training Pipeline (RF + Calibration)... This may take a while ⏳")
model_pipeline.fit(X_train, y_train)


🌲 Training Pipeline (RF + Calibration)... This may take a while ⏳


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",RandomForestC...ndom_state=42)
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",

In [14]:
# =========================
# 8️⃣ Evaluation (التقييم المعدل)
# =========================
print("\n📊 Evaluating model performance...")

y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)

# 1. المقاييس العامة
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print(f"✅ Accuracy  (الدقة العامة): {round(accuracy * 100, 2)} %")
print(f"✅ Precision (الدقة الموزونة): {round(precision * 100, 2)} %")
print(f"✅ Recall    (الاستدعاء الموزون): {round(recall * 100, 2)} %")
print(f"✅ F1-Score  (المقياس التوافقي): {round(f1 * 100, 2)} %")

# 2. تقرير مفصل (تم إصلاح الخطأ هنا) ⭐
print("\n📝 Detailed Classification Report:")

# نحدد فقط الفئات الموجودة فعلياً في مجموعة الاختبار
unique_labels = np.unique(y_test)

print(classification_report(
    y_test, 
    y_pred, 
    labels=unique_labels,  # نحدد الفئات المتاحة فقط
    target_names=label_encoder.classes_[unique_labels], # نجلب أسماء هذه الفئات فقط
    zero_division=0
))

# 3. حساب Brier Score
try:
    # نحتاج هنا للتأكد من توافق الأبعاد، لذا نستخدم معالجة الأخطاء فقط للحماية
    brier = np.mean([
        brier_score_loss((y_test == i).astype(int), y_prob[:, i])
        for i in unique_labels # نحسب فقط للفئات الموجودة
    ])
    print("📉 Mean Brier Score (Lower is better):", round(brier, 4))
except Exception as e:
    print(f"⚠️ Brier Score calculation skipped due to class mismatch: {e}")



📊 Evaluating model performance...
✅ Accuracy  (الدقة العامة): 77.82 %
✅ Precision (الدقة الموزونة): 79.6 %
✅ Recall    (الاستدعاء الموزون): 77.82 %
✅ F1-Score  (المقياس التوافقي): 77.84 %

📝 Detailed Classification Report:
                                                          precision    recall  f1-score   support

                               abdominal aortic aneurysm       0.53      1.00      0.70         8
                                        abdominal hernia       0.98      0.98      0.98        53
                                         abscess of nose       0.67      0.67      0.67        30
                                     abscess of the lung       1.00      1.00      1.00         1
                                  abscess of the pharynx       0.88      0.67      0.76        33
                                    acanthosis nigricans       0.00      0.00      0.00         2
                                               acariasis       1.00      1.00      1.00  

In [15]:
# =========================
# 9️⃣ Save Pipeline & Objects (حفظ النموذج)
# =========================
# حفظ البايبلاين بالكامل بدلاً من النموذج فقط
joblib.dump(model_pipeline, "disease_prediction_pipeline.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(X.columns.tolist(), "symptoms_list.pkl")
joblib.dump(df, "dataset_cleaned.pkl")

print("\n🎉 Pipeline trained and saved successfully!")
print("📁 Saved files:")
print(" - disease_prediction_pipeline.pkl (Contains Calibrated RF)")
print(" - label_encoder.pkl")
print(" - symptoms_list.pkl")
print(" - dataset_cleaned.pkl")



🎉 Pipeline trained and saved successfully!
📁 Saved files:
 - disease_prediction_pipeline.pkl (Contains Calibrated RF)
 - label_encoder.pkl
 - symptoms_list.pkl
 - dataset_cleaned.pkl
